# GDP / Population / Distance merge — 2015-2016 and 2024-2025

Fills in the gravity columns (`gdp_o`, `gdp_d`, `pop_o`, `pop_d`, `gdpcap_o`, `gdpcap_d`, `dist`)
for the two new Comtrade slices, which came from raw exports with these columns present but
empty (CEPII's Gravity dataset only covers through ~2020-2021, so it never had these years to
begin with).

- **GDP/population** -- pulled fresh from the World Bank API (`NY.GDP.MKTP.CD`, `SP.POP.TOTL`),
  the real published source, not modeled/estimated.
- **Distance** -- carried forward from the existing 2017-2023 file. Bilateral distance is
  time-invariant, so this is a straight lookup join on `(reporterCode, partnerCode)`, no year
  dimension needed.

**I can't run the actual World Bank API call from this environment** (no live network access
here) -- this notebook is built and its logic validated against a mocked response, but the
real HTTP call needs to happen on your machine. Watch the printed missing-value counts on
first run; if coverage looks thin for either GDP or population, that's the signal to check
before trusting anything downstream.

In [2]:
import pandas as pd
import numpy as np
import requests
import os
import sys

PROCESSED_DIR = os.path.join("..", "..", "data", "processed")
UTILS_DIR = os.path.join("../", "..", "src", "utils")
sys.path.append(UTILS_DIR)
from country_codes import ISO3_2_M49

iso2m49 = {**ISO3_2_M49, "TWN": "490"}

SLICES = [
    dict(tag="2015_2016", path=os.path.join(PROCESSED_DIR, "all_products_2015_2016_ready.parquet"),
         year_lo=2015, year_hi=2016),
    dict(tag="2024_2025", path=os.path.join(PROCESSED_DIR, "all_products_2024_2025_ready.parquet"),
         year_lo=2024, year_hi=2025),
]

DIST_SOURCE = os.path.join(PROCESSED_DIR, "all_products_ready.parquet")   # existing 2017-2023 file, for the dist lookup

## World Bank API pull -- GDP and population, per year range needed

In [3]:
def get_wb_indicator(indicator, year_start, year_end):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}"
    resp = requests.get(url, params={"format": "json", "date": f"{year_start}:{year_end}", "per_page": 20000})
    resp.raise_for_status()
    data = resp.json()[1]
    rows = [{"iso3": d["countryiso3code"], "year": int(d["date"]), "value": d["value"]}
            for d in data if d["value"] is not None]
    df = pd.DataFrame(rows)
    df["reporterCode"] = df["iso3"].map(iso2m49)
    df = df.dropna(subset=["reporterCode"])
    df["reporterCode"] = df["reporterCode"].astype(int)
    return df


all_years_lo = min(s["year_lo"] for s in SLICES)
all_years_hi = max(s["year_hi"] for s in SLICES)
print(f"pulling GDP/population for {all_years_lo}-{all_years_hi} (covers both slices in one call)")

gdp = get_wb_indicator("NY.GDP.MKTP.CD", all_years_lo, all_years_hi)
pop = get_wb_indicator("SP.POP.TOTL", all_years_lo, all_years_hi)

print(f"gdp: {len(gdp):,} rows, {gdp['reporterCode'].nunique()} countries, years {sorted(gdp['year'].unique())}")
print(f"pop: {len(pop):,} rows, {pop['reporterCode'].nunique()} countries, years {sorted(pop['year'].unique())}")

if gdp.empty or pop.empty:
    print("\nWARNING: World Bank pull returned empty -- check network access / API status before continuing.")

pulling GDP/population for 2015-2025 (covers both slices in one call)
gdp: 2,197 rows, 204 countries, years [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
pop: 2,288 rows, 208 countries, years [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## Distance lookup -- carried forward from the existing 2017-2023 file, no year dimension

In [4]:
dist_lookup = pd.read_parquet(DIST_SOURCE, columns=["reporterCode", "partnerCode", "dist"])
# cast to numeric explicitly -- the source parquet stores these as strings (the whole
# pipeline uses dtype=str during extraction), while merge_gravity() converts df's own
# reporterCode/partnerCode to numeric first. Without this, the merge fails with
# "You are trying to merge on int64 and object columns."
dist_lookup["reporterCode"] = pd.to_numeric(dist_lookup["reporterCode"], errors="coerce")
dist_lookup["partnerCode"] = pd.to_numeric(dist_lookup["partnerCode"], errors="coerce")
dist_lookup = dist_lookup.dropna(subset=["reporterCode", "partnerCode"])
dist_lookup["reporterCode"] = dist_lookup["reporterCode"].astype(int)
dist_lookup["partnerCode"] = dist_lookup["partnerCode"].astype(int)
dist_lookup = dist_lookup.drop_duplicates(["reporterCode", "partnerCode"])
print(f"distance lookup: {len(dist_lookup):,} unique reporter-partner pairs")

distance lookup: 15,507 unique reporter-partner pairs


## Merge function -- GDP/pop (origin + destination sides) + distance, with coverage report

In [5]:
def merge_gravity(df, gdp, pop, dist_lookup):
    df = df.copy()
    for c in ["reporterCode", "partnerCode", "refYear"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # origin (reporter) side
    df = df.merge(
        gdp.rename(columns={"value": "gdp_o"})[["reporterCode", "year", "gdp_o"]],
        left_on=["reporterCode", "refYear"], right_on=["reporterCode", "year"], how="left"
    ).drop(columns="year")
    df = df.merge(
        pop.rename(columns={"value": "pop_o"})[["reporterCode", "year", "pop_o"]],
        left_on=["reporterCode", "refYear"], right_on=["reporterCode", "year"], how="left"
    ).drop(columns="year")

    # destination (partner) side -- same WB tables, joined on partnerCode this time
    gdp_d = gdp.rename(columns={"value": "gdp_d", "reporterCode": "partnerCode"})[["partnerCode", "year", "gdp_d"]]
    pop_d = pop.rename(columns={"value": "pop_d", "reporterCode": "partnerCode"})[["partnerCode", "year", "pop_d"]]
    df = df.merge(gdp_d, left_on=["partnerCode", "refYear"], right_on=["partnerCode", "year"], how="left").drop(columns="year")
    df = df.merge(pop_d, left_on=["partnerCode", "refYear"], right_on=["partnerCode", "year"], how="left").drop(columns="year")

    df["gdpcap_o"] = df["gdp_o"] / df["pop_o"]
    df["gdpcap_d"] = df["gdp_d"] / df["pop_d"]

    # distance -- time-invariant, joined on country pair only
    df = df.merge(dist_lookup, on=["reporterCode", "partnerCode"], how="left", suffixes=("", "_lookup"))
    if "dist_lookup" in df.columns:
        df["dist"] = df["dist"].fillna(df["dist_lookup"])
        df = df.drop(columns="dist_lookup")

    return df

## Run the merge for both slices

In [6]:
results = []

for s in SLICES:
    tag, path = s["tag"], s["path"]
    print(f"\n{'='*50}\n{tag}\n{'='*50}")

    if not os.path.exists(path):
        print(f"  SKIPPED -- file not found: {path}")
        results.append(dict(tag=tag, status="missing"))
        continue

    df = pd.read_parquet(path)
    n_before = len(df)

    merged = merge_gravity(df, gdp, pop, dist_lookup)

    missing_gdp_o = merged["gdp_o"].isna().sum()
    missing_gdp_d = merged["gdp_d"].isna().sum()
    missing_pop_o = merged["pop_o"].isna().sum()
    missing_pop_d = merged["pop_d"].isna().sum()
    missing_dist = merged["dist"].isna().sum()
    any_missing = merged[["gdp_o", "gdp_d", "pop_o", "pop_d", "dist"]].isna().any(axis=1).sum()

    print(f"  rows: {n_before:,}")
    print(f"  missing: gdp_o={missing_gdp_o:,}  gdp_d={missing_gdp_d:,}  "
          f"pop_o={missing_pop_o:,}  pop_d={missing_pop_d:,}  dist={missing_dist:,}")
    print(f"  rows with at least one missing gravity value: {any_missing:,} ({100*any_missing/max(n_before,1):.1f}%)")

    out_path = path.replace("_ready.parquet", "_ready_gravity.parquet")
    merged.to_parquet(out_path, index=False)
    print(f"  -> {out_path}")

    results.append(dict(tag=tag, status="ok", rows=n_before, missing_any=any_missing, out=out_path))

summary = pd.DataFrame(results)
print("\n" + summary.to_string(index=False))


2015_2016
  rows: 10,147
  missing: gdp_o=1,324  gdp_d=848  pop_o=1,324  pop_d=795  dist=4,186
  rows with at least one missing gravity value: 5,837 (57.5%)
  -> ../../data/processed/all_products_2015_2016_ready_gravity.parquet

2024_2025
  rows: 21,317
  missing: gdp_o=2,002  gdp_d=2,118  pop_o=1,991  pop_d=1,498  dist=10,375
  rows with at least one missing gravity value: 13,249 (62.2%)
  -> ../../data/processed/all_products_2024_2025_ready_gravity.parquet

      tag status  rows  missing_any                                                               out
2015_2016     ok 10147         5837 ../../data/processed/all_products_2015_2016_ready_gravity.parquet
2024_2025     ok 21317        13249 ../../data/processed/all_products_2024_2025_ready_gravity.parquet


## What to check before trusting these files

- **If `missing_dist` is nonzero for any row**, that reporter-partner pair never appeared in
  the 2017-2023 file either -- likely a country pair with genuinely no prior trade history in
  your dataset, not a bug. Worth a spot-check on which pairs those are before dropping them.
- **If `missing_gdp_o`/`missing_pop_o` etc. are nonzero**, the World Bank simply hadn't
  published that country's figure for that year at pull time -- common for small
  economies/territories, and for the most recent year (2025) if pulled early in the year.
- These `_ready_gravity.parquet` files are what `test_bilateral_pairs.py` / `corridor_timeseries.py`
  should be pointed at for the 2015-2016 and 2024-2025 evaluation runs -- not the plain
  `_ready.parquet` files, which still have empty gravity columns.

In [7]:
merged_2015_2016 = pd.read_parquet("../../data/processed/all_products_2015_2016_ready_gravity.parquet")
merged_2024_2025 = pd.read_parquet("../../data/processed/all_products_2024_2025_ready_gravity.parquet")

for name, df in [("2015-2016", merged_2015_2016), ("2024-2025", merged_2024_2025)]:
    print(f"\n=== {name} ===")
    missing = df[df[["gdp_o","gdp_d","pop_o","pop_d","dist"]].isna().any(axis=1)]
    print("top reporterCode among missing rows:")
    print(missing["reporterCode"].value_counts().head(10))
    print("top partnerCode among missing rows:")
    print(missing["partnerCode"].value_counts().head(10))
    print("does partnerCode==490 (Taiwan) show up in missing rows?",
          (missing["partnerCode"] == 490).sum(), "rows")
    print("does reporterCode==490 (Taiwan) show up in missing rows?",
          (missing["reporterCode"] == 490).sum(), "rows")


=== 2015-2016 ===
top reporterCode among missing rows:
reporterCode
826    591
490    516
203    453
699    453
246    420
458    357
842    355
36     330
642    261
300    176
Name: count, dtype: int64
top partnerCode among missing rows:
partnerCode
842    186
56     116
699    111
490    104
276     74
591     69
724     67
528     67
380     66
251     65
Name: count, dtype: int64
does partnerCode==490 (Taiwan) show up in missing rows? 104 rows
does reporterCode==490 (Taiwan) show up in missing rows? 516 rows

=== 2024-2025 ===
top reporterCode among missing rows:
reporterCode
276    663
842    654
56     579
792    553
826    551
699    541
724    529
757    507
203    450
752    440
Name: count, dtype: int64
top partnerCode among missing rows:
partnerCode
842    297
699    218
56     203
490    193
784    176
591    153
528    146
156    145
276    139
344    136
Name: count, dtype: int64
does partnerCode==490 (Taiwan) show up in missing rows? 193 rows
does reporterCode==490 (Ta

In [8]:
print(gdp[gdp.reporterCode == 842])   # is USA actually in the pulled WB data at all?
print(gdp[gdp.reporterCode == 276])   # same for Germany

Empty DataFrame
Columns: [iso3, year, value, reporterCode]
Index: []
     iso3  year         value  reporterCode
1280  DEU  2025  5.050923e+12           276
1281  DEU  2024  4.685593e+12           276
1282  DEU  2023  4.562208e+12           276
1283  DEU  2022  4.201022e+12           276
1284  DEU  2021  4.355252e+12           276
1285  DEU  2020  3.941399e+12           276
1286  DEU  2019  3.959895e+12           276
1287  DEU  2018  4.055433e+12           276
1288  DEU  2017  3.765352e+12           276
1289  DEU  2016  3.536788e+12           276
1290  DEU  2015  3.425100e+12           276


In [9]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

def rf_fill_gravity(df, cols=["gdp_o","gdp_d","pop_o","pop_d","gdpcap_o","gdpcap_d","dist"]):
    imp = IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=0),
                           max_iter=5, random_state=0)
    df = df.copy()
    df[cols] = imp.fit_transform(df[cols])
    return df

merged_2015_2016 = rf_fill_gravity(merged_2015_2016)
merged_2024_2025 = rf_fill_gravity(merged_2024_2025)
merged_2015_2016.to_parquet("../../data/processed/all_products_2015_2016_ready_gravity.parquet", index=False)
merged_2024_2025.to_parquet("../../data/processed/all_products_2024_2025_ready_gravity.parquet", index=False)

/home/hefouzinho/miniconda3/envs/mathematical_implementations/lib/python3.8/site-packages/sklearn/impute/_iterative.py:800: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [10]:
existing = pd.read_parquet("../../data/processed/all_products_ready.parquet")

def taiwan_lookup_year(existing, anchor_year, code_col_is_reporter):
    """code_col_is_reporter=True -> Taiwan as origin (gdp_o/pop_o); False -> Taiwan as destination (gdp_d/pop_d)"""
    code_col = "reporterCode" if code_col_is_reporter else "partnerCode"
    sub = existing[existing[code_col].astype(str) == "490"]
    sub = sub[pd.to_numeric(sub["refYear"], errors="coerce") == anchor_year]
    if sub.empty:
        return None
    gdp_col, pop_col = ("gdp_o", "pop_o") if code_col_is_reporter else ("gdp_d", "pop_d")
    return sub[gdp_col].iloc[0], sub[pop_col].iloc[0]


def patch_taiwan(df, existing, anchor_year):
    df = df.copy()
    is_taiwan_reporter = df["reporterCode"].astype(str) == "490"
    is_taiwan_partner = df["partnerCode"].astype(str) == "490"

    r = taiwan_lookup_year(existing, anchor_year, code_col_is_reporter=True)
    if r is not None:
        gdp_o_fill, pop_o_fill = r
        df.loc[is_taiwan_reporter & df["gdp_o"].isna(), "gdp_o"] = gdp_o_fill
        df.loc[is_taiwan_reporter & df["pop_o"].isna(), "pop_o"] = pop_o_fill

    p = taiwan_lookup_year(existing, anchor_year, code_col_is_reporter=False)
    if p is not None:
        gdp_d_fill, pop_d_fill = p
        df.loc[is_taiwan_partner & df["gdp_d"].isna(), "gdp_d"] = gdp_d_fill
        df.loc[is_taiwan_partner & df["pop_d"].isna(), "pop_d"] = pop_d_fill

    df["gdpcap_o"] = df["gdp_o"] / df["pop_o"]
    df["gdpcap_d"] = df["gdp_d"] / df["pop_d"]
    return df

merged_2015_2016 = patch_taiwan(merged_2015_2016, existing, anchor_year=2017)
merged_2024_2025 = patch_taiwan(merged_2024_2025, existing, anchor_year=2023)

for name, df in [("2015-2016", merged_2015_2016), ("2024-2025", merged_2024_2025)]:
    still_missing = df[["gdp_o","gdp_d","pop_o","pop_d"]].isna().any(axis=1).sum()
    print(f"{name}: {still_missing:,} rows still missing gdp/pop after Taiwan patch")

2015-2016: 0 rows still missing gdp/pop after Taiwan patch
2024-2025: 0 rows still missing gdp/pop after Taiwan patch


In [11]:
for name, df in [("2015-2016", merged_2015_2016), ("2024-2025", merged_2024_2025)]:
    print(name, "dist missing:", df["dist"].isna().sum())

2015-2016 dist missing: 0
2024-2025 dist missing: 0


In [12]:
def patch_taiwan_dist(df, existing, anchor_year):
    df = df.copy()
    sub = existing[pd.to_numeric(existing["refYear"], errors="coerce") == anchor_year]
    dist_lookup_anchor = sub[["reporterCode","partnerCode","dist"]].drop_duplicates(["reporterCode","partnerCode"])
    dist_lookup_anchor["reporterCode"] = pd.to_numeric(dist_lookup_anchor["reporterCode"], errors="coerce").astype("Int64")
    dist_lookup_anchor["partnerCode"] = pd.to_numeric(dist_lookup_anchor["partnerCode"], errors="coerce").astype("Int64")
    df = df.merge(dist_lookup_anchor, on=["reporterCode","partnerCode"], how="left", suffixes=("", "_anchor"))
    df["dist"] = df["dist"].fillna(df["dist_anchor"])
    return df.drop(columns="dist_anchor")

merged_2015_2016 = patch_taiwan_dist(merged_2015_2016, existing, anchor_year=2017)
merged_2024_2025 = patch_taiwan_dist(merged_2024_2025, existing, anchor_year=2023)